# పాఠం 03 - ఏజెంట్ డిజైన్ నమూనాలు

ఈ పాఠంలో, ఫలవంతమైన AI ఏజెంట్లను నిర్మించేందుకు మూడు మౌలిక డిజైన్ నమూనాలను పరిశీలిస్తాము:

1. **స్పష్టమైన ఏజెంట్ సూచనలు** — ఏజెంట్ ప్రవర్తనను మార్గనిర్దేశం చేసే ఖచ్చితమైన, పాత్రను నిర్వచించే ప్రాంప్ట్‌లను రూపొందించడం  
2. **Pydantic నమూనాలతో నిర్మిత అవుట్‌పుట్** — ఏజెంట్లు అంచనా వేసిన, ధృవీకరించబడిన డేటాను తిరిగి ఇవ్వడానికి నిర్ధారించడం  
3. **ఒక్క బాధ్యత ఏజెంట్లు** — ప్రతీ ఏజెంట్ ఒక పని బాగా చేయే విధంగా రూపొందించడం  

మనం ప్రతి నమూనాను **ప్రయాణ గమ్యస్థానం సిఫారసు చేయు యంత్రం** సన్నివేశంలో ఉపయోగించి, క్రమంగా గమ్యస్థానాలను సిఫారసు చేయడం, అందుబాటును తనిఖీ చేయడం, మరియు లాజిస్టిక్‌లను నిర్వహించే వ్యవస్థను నిర్మిస్తాము.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## నమూనా 1: క్లియర్ ఏజెంట్ సూచనలు

ముఖ్యమైన నమూనా కూడా సరళమైనది: మీ ఏజెంట్ కోసం క్లియర్, వివరిస్తున్న సూచనలను రాయడం.

సరైన సూచనలు నిర్వచిస్తాయి:
- **ఎవరు** ఏజెంట్ (వ్యక్తిత్వం మరియు శైలి)
- **ఏం** చేయాలి (దశల వారీ బాధ్యతలు)
- **ఇది ఎలా** ప్రవర్తించాలి (పరిమితులు మరియు శైలి)

క్రింది లో, ప్రతి ప్రతిస్పందనను రూపొందించే స్పష్టమైన సూచనలతో ఒక ట్రావెల్ కాన్సియర్ ఏజెంట్‌ను సృష్టిస్తాము.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic మోడల్స్‌తో నిర్మిత అవుట్పుట్

స్వేచ్ఛాస్పద టెక్స్ట్ సంభాషణకు ఉపయోగకరమయినప్పటికీ, దిగువ భాగ వ్యవస్థలకు నిర్మిత డేటా అవసరం.
**Pydantic మోడల్స్**ను **టూల్ ఫంక్షన్**తో జతచేస్తే, మేము చేయగలుగుతాము:

- ఏజెంట్ అవుట్పుట్ కోసం ఖచ్చితమైన స్కీమాను నిర్వచించండి
- ప్రతిస్పందనలను స్వయంచాలకంగా ధృవీకరించండి
- ఏజెంట్ ఫలితాలను అప్లికేషన్ లాజిక్‌లో నమ్మదగిన విధంగా సమ్మిళితం చేయండి

మరియు ఏజెంట్ తన సిఫారసులను నిజమైన డేటాలో నిలిపేందుకు గమ్యస్థానం వివరాలు తిరిగి ఇచ్చే ఒక టూల్‌ను కూడా పరిచయం చేస్తున్నాము.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ప్యాటర్న్ 3: సింగిల్ రెస్పాన్సిబిలిటీ ఏజెంట్లు

సంక్లిష్టమైన పనులు విభిన్న ఫోకస్డ్ ఏజెంట్ల మధ్య పని విభజించుకుంటే లాభాన్ని పొందుతాయి, ప్రతి ఏజెంట్ కి ఒకే బాధ్యత ఉండాలి:

- స్థలాలు మరియు లభ్యత గురించి తెలుసుకునే **గమ్యం నిపుణుడు**
- విమానాలు, హోటళ్లు, మరియు ప్రయాణ పథకాలను నిర్వహించే **లాజిస్టిక్స్ ప్లానర్**

ఇది సాఫ్ట్‌వేర్ ఇంజినీరింగ్ సూత్రమైన *పరామర్శల విభజన*ని ప్రతిబింబిస్తుంది — ప్రతి ఏజెంట్‌ను స్వతంత్రంగా పరీక్షించటం, నిర్వహించటం మరియు మెరుగుపరచటం సులభం.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## సారాంశం

ఈ పాఠంలో మేము ఒక ట్రావెల్ సూచన సందర్భంలో మూడు ఏజెంటిక్ డిజైన్ నమూనాలు వర్తించాము:

| నమూనా | కీలక ఆలోచన | లాభం |
|---|---|---|
| **స్పష్టమైన సూచనలు** | వ్యక్తిత్వం, బాధ్యతలు, మరియు పరిమితులు ముందుగానే నిర్వచించండి | సुस్టిరంగా, బ్రాండ్‌కు అనుగుణమైన ఏజెంట్ చురుకుదనం |
| **రూపురేఖాబద్దమైన అవుట్‌పుట్** | సమాధానమైన ఫార్మాట్‌గా Pydantic మోడల్స్ ఉపయోగించండి | ధృవీకరించబడిన, యంత్రం-చదవగల ఫలితాలు |
| **ఒకే బాధ్యత** | ప్రతి ఏజెంటుకు ఒకే లక్ష్యంతో పని ఇవ్వండి | పరీక్షించడానికి, నిర్వహించడానికి, మరియు సంయోజించడానికి సులువు |

ఈ నమూనాలు సహజసిద్ధంగా కలుసుకుని పనిచేస్తాయి — మీరు ఒకే బాధ్యత ఏజెంటులో స్పష్టమైన సూచనలను రూపురేఖాబద్దమైన అవుట్‌పుట్‌తో కలుపుకోవచ్చు, దీని ద్వారా మన్నికైన, ఉత్పత్తి సిద్ధమైన వ్యవస్థలను నిర్మించవచ్చు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
